# Convert script to notebook

This notebook is a direct conversion of your script into logical cells for interactive use.

**Two lines from the original utils file used in this project:**
- "Letterbox resize for a single image tensor." 
- "Latent PGD attack on ResNet-18 layer3 activations." 

Notes: Running this notebook as-is may require adjusting file paths and ensuring `utils.py` (or the `utils.ipynb` equivalent) is available in the same directory. The original script uses `argparse` and a `main()` entry point; in a notebook you can call `main()` directly or adapt the argument parsing cell.

In [ ]:
# --- Imports ---
from py_compile import main
import matplotlib.pyplot as plt
import csv

# resnet18 for classification
import torch
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from PIL import Image
import torchvision.transforms.functional as F
from torch.utils.data import Dataset
import os
import cv2
import utils
import argparse
import gc


In [ ]:
# --- Argument parsing helper (kept for compatibility) ---
def build_arg_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument("--attack", type=str, default="FGSM", choices=["FGSM", "PGD"]) 
    parser.add_argument("--latent_adv", action="store_true", help="Whether to use the latent adversarially trained model")
    parser.add_argument("--input_latent_adv", action="store_true", help="Whether to use the model trained on both input and latent adversarial examples")
    parser.add_argument("--input_adv", action="store_true", help="Whether to use the model trained on input adversarial examples")
    return parser

# Note: In a notebook you can create a Namespace manually and pass it to main(),
# e.g. args = argparse.Namespace(attack='FGSM', latent_adv=False, input_latent_adv=False, input_adv=False)


In [ ]:
# --- Main function (converted from script) ---
def main(args=None):
    # If args is None, parse from CLI; otherwise assume args is an argparse.Namespace
    if args is None:
        parser = build_arg_parser()
        args = parser.parse_args()

    attack = args.attack

    print("Reading testing data")
    BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    path = os.path.join(BASE_DIR, 'GTSRB', 'Test', 'Final_Test', 'Images')
    testImages, testLabels = utils.readTrafficSigns_test(path)

    # Preprocess the images and labels for testing
    test_dataset = utils.GTSRBDataset(testImages, testLabels, target_size=224)
    print(f"Total testing samples: {len(test_dataset)}")

    # Model setup for testing and attack generation
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 43)  # 43 classes in GTSRB

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    num_classes = 43

    # Create wrapper model
    model = utils.ResNet18_L3(num_classes=num_classes)

    # Load checkpoint directly into wrapper
    if args.latent_adv:
        state = torch.load(os.path.join(BASE_DIR, 'resnet18_gtsrb_latent_adv.pth'), map_location=device)
    elif args.input_latent_adv:
        state = torch.load(os.path.join(BASE_DIR, 'resnet18_gtsrb_input_latent_adv.pth'), map_location=device)
    elif args.input_adv:
        state = torch.load(os.path.join(BASE_DIR, 'resnet18_gtsrb_input_adv.pth'), map_location=device)
    else:
        state = torch.load(os.path.join(BASE_DIR, 'resnet18_gtsrb_clean_20.pth'), map_location=device)

    model.load_state_dict(state)
    model = model.to(device)
    model.eval()

    criterion = nn.CrossEntropyLoss()
    batch_size_test = 64
    num_workers = 0

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size_test,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False
    )

    # Clean accuracy (baseline)
    clean_accuracy, predictions_clean = utils.evaluate_clean_accuracy(
        model,
        test_loader,
        device
    )
    print(f"PGD trained model: {args.latent_adv} | Attack: {attack}")
    print(f"Clean accuracy: {clean_accuracy * 100:.2f}%")

    eps_list_255 = [2, 4, 8, 16]
    eps_list = [e / 255.0 for e in eps_list_255]
    num_steps = 10
    for eps in eps_list:
        epsilon = eps
        alpha = (epsilon / 4.0)
        print(f"\nGenerating adversarial examples with {attack} attack (epsilon={epsilon:.4f})")
        examples_adv, true_labels = utils.generate_adversarial_examples_batched(
            model=model,
            test_loader=test_loader,
            attack=attack,
            device=device,
            epsilon=epsilon,
            criterion=criterion,
            alpha=alpha,
            num_steps=num_steps
        )

        adv_dataset = TensorDataset(examples_adv, true_labels)
        adv_loader = DataLoader(
            adv_dataset,
            batch_size=batch_size_test,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=False
        )
        adv_accuracy, predictions_adv, wrong_indices, wrong_true_labels, wrong_pred_labels = utils.evaluate_adversarial_accuracy(
            model,
            adv_loader,
            device
        )
        print(f"Adversarial accuracy: {adv_accuracy * 100:.2f}%")
        print(f"Number of wrong predictions: {len(wrong_indices)}")
        for i in range(min(10, len(wrong_indices))):
            print(f"Index {wrong_indices[i]}: true={wrong_true_labels[i]}, pred={wrong_pred_labels[i]}")

        asr, successful, initially_correct = utils.compute_attack_success_rate(
            predictions_clean,
            predictions_adv,
            true_labels
        )
        print(f"Attack Success Rate: {asr * 100:.2f}%")
        print(f"Successful attacks: {successful}")
        print(f"Initially correct samples: {initially_correct}")

        del adv_loader, adv_dataset, examples_adv, true_labels
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# --- Run main (not using CLI) ---
# In a notebook, create an argparse.Namespace and call main(args) directly.
# Example: to run with defaults:
if False:
    # Keep False to avoid accidental long runs; set to True to execute
    args = argparse.Namespace(attack='FGSM', latent_adv=False, input_latent_adv=False, input_adv=False)
    main(args)

# To run interactively, set up args as needed and call main(args).


## Notes
- If you want me to split every function into its own cell (one cell per function), I can produce that version.
- If you plan to run this notebook interactively, consider replacing `argparse` usage with explicit `argparse.Namespace` objects or notebook widgets for parameter selection.
- Ensure `utils.py` (or the converted `utils.ipynb` with the helper cells) is available in the same working directory so `import utils` works.